# Build a grounded Q/A Agent with ollama service with Langchain

## Setup

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 8.72 ms, sys: 10.7 ms, total: 19.4 ms
Wall time: 1.06 s


### OSS libraries install

In [2]:
%pip install wget sentence_transformers chromadb langchain langchain_chroma langchain-community  langchain-ollama pydantic sqlalchemy ipython-autotime --use-deprecated=legacy-resolver

Note: you may need to restart the kernel to use updated packages.


### My variables

In [3]:
my_model_ollama = "llama3.2"

## Load document data and build knowledge base

In [4]:
import requests
import os
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

# Define filename and URL
filename = "./docs/StateOfUnion.txt"
# Download the file if it does not exist

# Download the file if it does not exist
if not os.path.isfile(filename):
    response = requests.get(filename)
    with open(filename, "wb") as f:
        f.write(response.content)

# Load the document
loader = TextLoader(filename)
documents = loader.load()

# Split the document into chunks using CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(texts)}")

texts

Number of chunks: 42


[Document(metadata={'source': './docs/StateOfUnion.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russiaâ€™s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their deter

In [5]:
texts[0].page_content

'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russiaâ€™s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.'

## Create an embedding model

In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model=my_model_ollama
)

embeddings

OllamaEmbeddings(model='llama3.2', base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## Embed Documents and Store

In [7]:
from langchain.vectorstores import Chroma

docsearch = Chroma.from_documents(texts, embeddings)
docsearch

In [8]:
sample_texts = texts[:3]   # Taking a sample of 3 documents for demonstration
sample_texts

[Document(metadata={'source': './docs/StateOfUnion.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russiaâ€™s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their deter

In [9]:
sample_embeddings = embeddings.embed_documents([doc.page_content for doc in sample_texts])
sample_embeddings

[[0.0041456656,
  0.017982928,
  -0.011638774,
  -0.011547141,
  -0.012751507,
  -0.0069857263,
  0.010086349,
  0.00046649715,
  -0.01573384,
  1.866881e-05,
  0.0062862667,
  0.016461326,
  0.0135810515,
  0.029327393,
  0.004350988,
  -0.008132286,
  -0.0036518883,
  -0.018242758,
  0.0043208543,
  0.053692926,
  0.004427162,
  -0.0052855625,
  -0.001052744,
  -0.007820238,
  -0.020106088,
  -0.0086786235,
  0.016055863,
  -0.027048111,
  -0.0021319196,
  0.003878747,
  0.0061967224,
  0.018015653,
  0.011015649,
  0.015777575,
  -0.002416908,
  0.0042553856,
  0.014000874,
  -0.014609197,
  0.005346728,
  -0.005412686,
  -0.008370241,
  0.028124258,
  -0.0012867533,
  0.00042590583,
  -0.008740883,
  -0.001964612,
  -0.0019649707,
  0.018205373,
  0.02691785,
  -0.010786112,
  -0.011623281,
  0.019791542,
  0.007416607,
  -0.0023188302,
  0.015005605,
  0.01260295,
  0.0028706824,
  -0.037776057,
  0.009401924,
  0.0043180473,
  0.014779674,
  0.0069351885,
  -0.02899415,
  -0.0121

In [10]:
print("Sample Embedding Vectors:")
for i, embedding in enumerate(sample_embeddings):
    print(f"Document {i + 1} Embedding Vector: Length: {len(embedding)}; {embedding}")

Sample Embedding Vectors:
Document 1 Embedding Vector: Length: 3072; [0.0041456656, 0.017982928, -0.011638774, -0.011547141, -0.012751507, -0.0069857263, 0.010086349, 0.00046649715, -0.01573384, 1.866881e-05, 0.0062862667, 0.016461326, 0.0135810515, 0.029327393, 0.004350988, -0.008132286, -0.0036518883, -0.018242758, 0.0043208543, 0.053692926, 0.004427162, -0.0052855625, -0.001052744, -0.007820238, -0.020106088, -0.0086786235, 0.016055863, -0.027048111, -0.0021319196, 0.003878747, 0.0061967224, 0.018015653, 0.011015649, 0.015777575, -0.002416908, 0.0042553856, 0.014000874, -0.014609197, 0.005346728, -0.005412686, -0.008370241, 0.028124258, -0.0012867533, 0.00042590583, -0.008740883, -0.001964612, -0.0019649707, 0.018205373, 0.02691785, -0.010786112, -0.011623281, 0.019791542, 0.007416607, -0.0023188302, 0.015005605, 0.01260295, 0.0028706824, -0.037776057, 0.009401924, 0.0043180473, 0.014779674, 0.0069351885, -0.02899415, -0.012189546, 0.04984787, -0.015753709, 0.020827774, 0.0014648706

## LangChain Ollama model

In [11]:
# Initialize model parameters
my_params = {
        "decoding_method": "greedy",
        "temperature": 0.4, 
        "min_new_tokens": 1,
        "max_new_tokens": 100,
        #"stop_sequences":["\n"]
    }
my_params

{'decoding_method': 'greedy',
 'temperature': 0.4,
 'min_new_tokens': 1,
 'max_new_tokens': 100}

In [12]:
from langchain_ollama.llms import OllamaLLM

my_llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    stream=True,
    params=my_params
)

my_llm_client

OllamaLLM(model='llama3.2', base_url='http://localhost:11434')

## Generate a retrieval augmented response to a question

Question answering chain to automate the RAG pipeline.

In [13]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(llm=my_llm_client, chain_type="stuff", retriever=docsearch.as_retriever())
qa

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://localhost:11434'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), retriever=VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x142904b10>, search_kwargs={}))

## Question-answering agent

Answers the questions by using the RAG pipeline.

In [14]:
query = "What did the president say about highway and bridges in disrepair"
qa.invoke(query)

{'query': 'What did the president say about highway and bridges in disrepair',
 'result': 'The President mentioned that America used to have the best roads, bridges, and airports on Earth, but now its infrastructure is ranked 13th in the world.'}

In [15]:
query = "What did the president say about the infrastructure rank in the world?"
qa.invoke(query)

{'query': 'What did the president say about the infrastructure rank in the world?',
 'result': "The president stated that America's current infrastructure ranking is 13th in the world, a decline from when the country had the best roads, bridges, and airports on Earth."}

In [16]:
query = query = "What did the president say about a Unity Agenda for the Nation? What is the first thing we can do together?"
qa.invoke(query)

{'query': 'What did the president say about a Unity Agenda for the Nation? What is the first thing we can do together?',
 'result': 'The president stated that they have a unity agenda for the nation. The first thing we can do together is to build this national network of 500,000 electric vehicle charging stations.'}